clone the code and have env setup.

In [1]:
import os

REPO_URL = 'https://github.com/Lynx-Zhang/DD2424-Project.git'
REPO_DIR = '/content/DD2424-Project'
BRANCH = 'dev/AdamAndSentiment'
# BRANCH = 'main'


from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}

print(f"\n当前分支: ", end='')
!git branch --show-current



!mkdir -p predictions
!mkdir -p /content/drive/MyDrive/gpt2_logs
!mkdir -p /content/drive/MyDrive/gpt2_checkpoints

print("\n GPU 信息:")

print("\n 环境准备完成！")


Mounted at /content/drive
Cloning into '/content/DD2424-Project'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 100 (delta 32), reused 47 (delta 26), pack-reused 30 (from 1)
Receiving objects: 100% (100/100), 31.99 MiB | 19.87 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/DD2424-Project

当前分支: dev/AdamAndSentiment

 GPU 信息:

 环境准备完成！


For Sentiment Analysis Part we have

In [2]:
# 按照 cs224n_dfp 环境装依赖（pip 版本）
!pip install -q \
    tqdm==4.58.0 \
    requests==2.25.1 \
    importlib-metadata==3.7.0 \
    filelock==3.0.12 \
    tokenizers==0.20 \
    explainaboard_client==0.0.7 \
    einops==0.8.0 \
    transformers==4.46.3 \
    sacrebleu==2.5.1 \
    scikit-learn

# torch 不动（用 Colab 自带的 CUDA 版）
print(" 依赖安装完成")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 58.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 133.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 kB 3.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━

In [3]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-1257106e-690f-a6ea-ad99-0d87dfe928ec)


In [4]:
!pwd
! ls
%cd DD2424-Project
! ls


/content/DD2424-Project
classifier.py					 optimizer.py
config.py					 optimizer_test.npy
CS_224n__Default_Final_Project__Build_GPT_2.pdf  optimizer_test.py
data						 paraphrase_detection.py
datasets.py					 predictions
DD2424_Project.ipynb				 prepare_submit.py
doc						 README.md
env.yml						 sanity_check.py
evaluation.py					 setup.sh
LICENSE						 sonnet_generation.py
models						 utils.py
modules
[Errno 2] No such file or directory: 'DD2424-Project'
/content/DD2424-Project
classifier.py					 optimizer.py
config.py					 optimizer_test.npy
CS_224n__Default_Final_Project__Build_GPT_2.pdf  optimizer_test.py
data						 paraphrase_detection.py
datasets.py					 predictions
DD2424_Project.ipynb				 prepare_submit.py
doc						 README.md
env.yml						 sanity_check.py
evaluation.py					 setup.sh
LICENSE						 sonnet_generation.py
models						 utils.py
modules


In [5]:
# 跑 last-linear-layer 模式
!python classifier.py --use_gpu --fine-tune-mode last-linear-layer --epochs 10 --lr 1e-3 --batch_size 64
# 备份到 Drive
!cp -r predictions /content/drive/MyDrive/gpt2_checkpoints/last-linear-$(date +%H%M%S)



2026-05-02 20:16:29.010699: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Training Sentiment Classifier on SST...
load 8544 data from data/ids-sst-train.csv
load 1101 data from data/ids-sst-dev.csv
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 106kB/s]
vocab.json: 1.04MB [00:00, 10.2MB/s]
merges.txt: 456kB [00:00, 19.9MB/s]
tokenizer.json: 1.36MB [00:00, 5.66MB/s]
config.json: 100% 665/665 [00:00<00:00, 4.94MB/s]
model.safetensors: 100% 548M/548M [00:02<00:00, 222MB/s] 
train-0: 100% 134/134 [00:22<00:00,  5.86it/s]
eval: 100% 134/134 [00:21<00:00,  6.18it/s]
eval: 100% 18/18 [00:02<00:00,  6.02it/s]
save the model to sst-classifier.pt
Epoch 0: train loss :: 1.656, train acc :: 0.416, dev acc :: 0.406
train-1: 100% 134/134 [00:22<00:00,  

In [6]:
# 跑 full-model 模式
!python classifier.py --use_gpu --fine-tune-mode full-model --epochs 10 --lr 1e-5 --batch_size 16

# 备份到 Drive
!cp -r predictions /content/drive/MyDrive/gpt2_checkpoints/full-model-$(date +%H%M%S)

2026-05-02 20:40:52.417748: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Training Sentiment Classifier on SST...
load 8544 data from data/ids-sst-train.csv
load 1101 data from data/ids-sst-dev.csv
train-0: 100% 534/534 [01:35<00:00,  5.61it/s]
eval: 100% 534/534 [00:22<00:00, 24.20it/s]
eval: 100% 69/69 [00:03<00:00, 22.48it/s]
save the model to sst-classifier.pt
Epoch 0: train loss :: 1.492, train acc :: 0.496, dev acc :: 0.452
train-1: 100% 534/534 [01:34<00:00,  5.63it/s]
eval: 100% 534/534 [00:22<00:00, 23.97it/s]
eval: 100% 69/69 [00:02<00:00, 25.06it/s]
save the model to sst-classifier.pt
Epoch 1: train loss :: 1.182, train acc :: 0.566, dev acc :: 0.513
train-2: 100% 534/534 [01:34<00:00,  5.64it/s]
eval: 100% 534/534 [00:21<00:00, 24.3

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
